In [1]:
import numpy as np
import pandas as pd

# 1. Matriks Jarak sesuai gambar (d_ij)
cities = ['A', 'B', 'C', 'D', 'E']
dist_matrix = np.array([
    [0, 10, 12, 11, 14],
    [10, 0, 13, 15, 8],
    [12, 13, 0, 9, 14],
    [11, 15, 9, 0, 16],
    [14, 8, 14, 16, 0]
])

# 2. Parameter ACO
alpha = 1
beta = 2
rho = 0.1
pheromone_initial = 1.0

# 3. Matriks Visibilitas (1/d_ij) dan Feromon Awal
visibility = 1.0 / np.where(dist_matrix == 0, np.inf, dist_matrix)
pheromone = np.ones((5, 5))
np.fill_diagonal(pheromone, 0)

print("Setup Data Berhasil.")
pd.DataFrame(dist_matrix, index=cities, columns=cities)

Setup Data Berhasil.


,A,B,C,D,E
A,0,10,12,11,14
B,10,0,13,15,8
C,12,13,0,9,14
D,11,15,9,0,16
E,14,8,14,16,0


In [2]:
# Masukkan Angka Random (R) sesuai pengerjaan Excel
R_values = {
    "Semut 1": [0.6841, 0.4024, 0.5252],
    "Semut 2": [0.1250, 0.7530, 0.2210],
    "Semut 3": [0.4820, 0.1150, 0.8940],
    "Semut 4": [0.9120, 0.3460, 0.6710]
}

# Rute hasil seleksi berdasarkan R tersebut
rute_semut = {
    "Semut 1": ['A', 'D', 'C', 'E', 'B', 'A'],
    "Semut 2": ['A', 'B', 'E', 'C', 'D', 'A'],
    "Semut 3": ['A', 'C', 'D', 'E', 'B', 'A'],
    "Semut 4": ['A', 'E', 'B', 'D', 'C', 'A']
}

print("Data Rute dan Angka Random R telah disiapkan.")

Data Rute dan Angka Random R telah disiapkan.


In [3]:
delta_tau_total = np.zeros((5, 5))
summary_data = []

print("=== DETAIL PERHITUNGAN TIAP SEMUT ===\n")

for name, rute in rute_semut.items():
    # A. Hitung Jarak (L)
    L = 0
    path_detail = []
    for i in range(len(rute) - 1):
        idx_a = cities.index(rute[i])
        idx_b = cities.index(rute[i+1])
        d = dist_matrix[idx_a][idx_b]
        L += d
        path_detail.append(f"{rute[i]}->{rute[i+1]} ({d})")
    
    # B. Hitung Delta Tau (1/L)
    dt = 1 / L
    summary_data.append([name, "-".join(rute), L, dt])
    
    # C. Simulasi Logika Tahap 1 (A ke Tujuan)
    idx_start = 0 # Kota A
    R1 = R_values[name][0]
    nums = [(pheromone[idx_start, t]**alpha) * (visibility[idx_start, t]**beta) for t in range(1, 5)]
    probs = [n/sum(nums) for n in nums]
    cum_probs = np.cumsum(probs)
    
    print(f"[{name}]")
    print(f"  Rute: {' + '.join(path_detail)} = {L}")
    print(f"  Tahap 1: R={R1} | Akumulasi Prob: B:{cum_probs[0]:.3f}, C:{cum_probs[1]:.3f}, D:{cum_probs[2]:.3f}, E:{cum_probs[3]:.3f}")
    print(f"  Keputusan Tahap 1: Pilih {rute[1]}\n")
    
    # D. Akumulasi tambahan feromon untuk Matriks Baru (Simetris)
    for i in range(len(rute) - 1):
        ia, ib = cities.index(rute[i]), cities.index(rute[i+1])
        delta_tau_total[ia][ib] += dt
        delta_tau_total[ib][ia] += dt

print("Perhitungan Jarak Selesai.")

=== DETAIL PERHITUNGAN TIAP SEMUT ===

[Semut 1]
  Rute: A->D (11) + D->C (9) + C->E (14) + E->B (8) + B->A (10) = 52
  Tahap 1: R=0.6841 | Akumulasi Prob: B:0.330, C:0.559, D:0.832, E:1.000
  Keputusan Tahap 1: Pilih D

[Semut 2]
  Rute: A->B (10) + B->E (8) + E->C (14) + C->D (9) + D->A (11) = 52
  Tahap 1: R=0.125 | Akumulasi Prob: B:0.330, C:0.559, D:0.832, E:1.000
  Keputusan Tahap 1: Pilih B

[Semut 3]
  Rute: A->C (12) + C->D (9) + D->E (16) + E->B (8) + B->A (10) = 55
  Tahap 1: R=0.482 | Akumulasi Prob: B:0.330, C:0.559, D:0.832, E:1.000
  Keputusan Tahap 1: Pilih C

[Semut 4]
  Rute: A->E (14) + E->B (8) + B->D (15) + D->C (9) + C->A (12) = 58
  Tahap 1: R=0.912 | Akumulasi Prob: B:0.330, C:0.559, D:0.832, E:1.000
  Keputusan Tahap 1: Pilih E

Perhitungan Jarak Selesai.


In [4]:
# 1. Rumus Update: (1-rho)*Tau_Lama + Delta_Tau_Total
pheromone_new = ((1 - rho) * pheromone) + delta_tau_total
np.fill_diagonal(pheromone_new, 0) # Diagonal tetap 0

# 2. Cetak Ringkasan
print("=== RINGKASAN HASIL SEMUT 1-4 ===")
df_res = pd.DataFrame(summary_data, columns=['Semut', 'Rute', 'Jarak (L)', 'Delta Tau'])
print(df_res.to_string(index=False))

print("\n=== MATRIKS FEROMON BARU (UNTUK ITERASI 2) ===")
df_final = pd.DataFrame(pheromone_new, index=cities, columns=cities)
print(df_final.round(5))

=== RINGKASAN HASIL SEMUT 1-4 ===
  Semut        Rute  Jarak (L)  Delta Tau
Semut 1 A-D-C-E-B-A         52   0.019231
Semut 2 A-B-E-C-D-A         52   0.019231
Semut 3 A-C-D-E-B-A         55   0.018182
Semut 4 A-E-B-D-C-A         58   0.017241

=== MATRIKS FEROMON BARU (UNTUK ITERASI 2) ===
         A        B        C        D        E
A  0.00000  0.95664  0.93542  0.93846  0.91724
B  0.95664  0.00000  0.90000  0.91724  0.97388
C  0.93542  0.90000  0.00000  0.97388  0.93846
D  0.93846  0.91724  0.97388  0.00000  0.91818
E  0.91724  0.97388  0.93846  0.91818  0.00000
